# Multi-Stage Vision System for Poultry Health

This notebook documents the experimentation process for training both detection stages.
Production training scripts have been moved to `training/train_stage1.py` and `training/train_stage2.py`.
Inference logic lives in `inference/pipeline.py`.

**Stage 1** - Binary detector (Normal vs AbNormal) trained on the Disease-Prediction dataset.

**Stage 2** - Fine-grained disease classifier (Avian Influenza, Fowl Pox, etc.) trained on the HenDiseaseDetection dataset.

## Setup

In [ ]:
import os

HOME = os.getcwd()

## Install Dependencies

Skip if already installed.

In [ ]:
# !pip install ultralytics==8.2.103 -q
# !pip install roboflow==1.1.48 --quiet

## Imports

In [ ]:
from IPython import display
display.clear_output()

from ultralytics import YOLO
from IPython.display import display, Image
import ultralytics
from roboflow import Roboflow

ultralytics.checks()

## Dataset Download

Set `ROBOFLOW_API_KEY_1` and `ROBOFLOW_API_KEY_2` as environment variables before running.

In [ ]:
os.makedirs(f"{HOME}/datasets", exist_ok=True)
%cd {HOME}/datasets

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY_1"])
project = rf.workspace("chicken-disease").project("disease-prediction-oryuo")
dataset = project.version(2).download("yolov8")

In [ ]:
rf2 = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY_2"])
project2 = rf2.workspace("obinson-tobinson").project("hendiseasedetection-octdy")
dataset2 = project2.version(1).download("yolov8")

## Stage 1 Training

Binary detection: Normal vs AbNormal.

Dataset: Disease-Prediction-2 (140 train / 40 val / 20 test)

Architecture: YOLOv8s, 30 epochs, imgsz=800

In [ ]:
!yolo task=detect mode=train model=yolov8s.pt data={dataset.location}/data.yaml epochs=30 imgsz=800 plots=True

### Stage 1 Results

In [ ]:
Image(filename=f"{HOME}/runs/detect/train/confusion_matrix.png", width=600)

In [ ]:
Image(filename=f"{HOME}/runs/detect/train/val_batch0_pred.jpg", width=600)

### Stage 1 Validation

In [ ]:
!yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

### Stage 1 Inference on Test Set

In [ ]:
!yolo task=detect mode=predict model={HOME}/runs/detect/train/weights/best.pt conf=0.60 source={dataset.location}/test/images save=True

In [ ]:
import glob

base_path = f"{HOME}/runs/detect/"
predict_dirs = sorted(
    [d for d in os.listdir(base_path) if d.startswith("predict")],
    key=lambda d: os.path.getmtime(os.path.join(base_path, d))
)
latest = os.path.join(base_path, predict_dirs[-1])

for path in glob.glob(f"{latest}/*.jpg")[:3]:
    display(Image(filename=path, width=600))

## Stage 2 Training

Fine-grained disease classification: Avian Influenza, Fowl Pox, Hen, etc.

Dataset: HenDiseaseDetection-1 (246 train / 31 val)

Architecture: YOLOv8s, 10 epochs, imgsz=800

In [ ]:
!yolo task=detect mode=train model=yolov8s.pt data={dataset2.location}/data.yaml epochs=10 imgsz=800 plots=True

### Stage 2 Results

In [ ]:
Image(filename=f"{HOME}/runs/detect/train3/confusion_matrix.png", width=600)

In [ ]:
Image(filename=f"{HOME}/runs/detect/train3/val_batch0_pred.jpg", width=600)

### Stage 2 Validation

In [ ]:
!yolo task=detect mode=val model={HOME}/runs/detect/train3/weights/best.pt data={dataset2.location}/data.yaml

### Stage 2 Inference on Test Set

In [ ]:
!yolo task=detect mode=predict model={HOME}/runs/detect/train3/weights/best.pt conf=0.60 source={dataset.location}/test/images save=True